In [2]:
import torch
import torch.nn as nn
import lab07_starter as S

In [3]:
class VAE(nn.Module):
    def __init__(self, d_in=64, d_hidden=64, d_latent=8):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(d_hidden, d_latent)
        self.fc_var = nn.Linear(d_hidden, d_latent)
        
        self.dec = nn.Sequential(
            nn.Linear(d_latent, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, d_in) # Returns logits for Bernoulli likelihood
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_var(h)

    def reparameterize(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

def vae_loss_fn(model, xb, yb=None, beta=1.0):
    recon_logits, mu, log_var = model(xb)
    
    recon_loss = nn.functional.binary_cross_entropy_with_logits(
        recon_logits, xb, reduction='none'
    ).sum(dim=1).mean()
    
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=1).mean()
    
    return recon_loss + beta * kl_loss

print("VAE class and loss function defined successfully inside the notebook!")

VAE class and loss function defined successfully inside the notebook!
